# 🛰️ Rondônia SR Study — Phase 1: Data Download & Patch Preparation

**This notebook (no GPU needed — uses CPU only, saves your weekly quota)**

Steps:
1. Install dependencies
2. Load project code from your uploaded dataset
3. Download 12 Landsat-8 + Sentinel-2 scene pairs from Microsoft Planetary Computer (~4–6 GB)
4. Sensor harmonization + sub-pixel alignment + patch extraction (~2700 patches)
5. Save patches as a Kaggle Dataset output for Phase 2

**Before running:** Make sure `Internet` is **ON** in Notebook Settings (right panel).

**Time estimate:** ~45–90 minutes on Kaggle CPU.

---
## Step 1 — Configuration
Set your Kaggle username here (same as the one you used to upload the code dataset).

In [ ]:
# ════════════════════════════════════════════════════
# EDIT THIS — your Kaggle username
KAGGLE_USER = "your_kaggle_username"
# ════════════════════════════════════════════════════

import os, sys
from pathlib import Path

# Working directories on Kaggle
WORK_DIR    = Path("/kaggle/working")
CODE_DIR    = Path(f"/kaggle/input/rondonia-sr-code")
DATA_DIR    = WORK_DIR / "data"
PATCHES_DIR = WORK_DIR / "data" / "aligned"
RAW_DIR     = WORK_DIR / "data" / "raw"

for d in [DATA_DIR, PATCHES_DIR, RAW_DIR,
          RAW_DIR / "landsat", RAW_DIR / "sentinel",
          PATCHES_DIR / "train", PATCHES_DIR / "val", PATCHES_DIR / "test"]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Working directory : {WORK_DIR}")
print(f"Code directory    : {CODE_DIR}")
print(f"Code dir exists   : {CODE_DIR.exists()}")
if CODE_DIR.exists():
    print(f"Files in code dir : {[f.name for f in CODE_DIR.iterdir()]}")

---
## Step 2 — Install Dependencies

In [ ]:
%%bash
pip install -q \
    rasterio>=1.3.9 \
    pystac-client>=0.7.0 \
    planetary-computer>=1.0.0 \
    stackstac>=0.4.4 \
    scikit-image>=0.22.0 \
    scipy>=1.11.0 \
    tqdm>=4.66.0 \
    pyyaml>=6.0.1 \
    numpy>=1.24.0 \
    pandas>=2.0.0
echo "All packages installed."

---
## Step 3 — Set Up Project Code
Copies code from your uploaded dataset into the working directory.

In [ ]:
import shutil, sys
from pathlib import Path

CODE_DIR = Path(f"/kaggle/input/rondonia-sr-code")
WORK_DIR = Path("/kaggle/working")

if not CODE_DIR.exists():
    raise FileNotFoundError(
        f"Code dataset not found at {CODE_DIR}.\n"
        "Please run push_to_kaggle.sh locally first, then:\n"
        "  1. Click '+ Add Data' in this notebook\n"
        "  2. Search for 'rondonia-sr-code' under Your Datasets\n"
        "  3. Add it and re-run this cell."
    )

# Copy code files to working directory so imports work
for item in CODE_DIR.iterdir():
    dest = WORK_DIR / item.name
    if item.is_dir():
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)
    print(f"  Copied: {item.name}")

# Add to Python path
sys.path.insert(0, str(WORK_DIR))
print("\n✓ Project code ready.")

---
## Step 4 — Patch config.yaml for Kaggle Paths
Updates the config to point to Kaggle's working directory.

In [ ]:
import yaml
from pathlib import Path

WORK_DIR = Path("/kaggle/working")
cfg_path = WORK_DIR / "config.yaml"

with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Override all paths for Kaggle
cfg["paths"]["landsat_dir"]   = str(WORK_DIR / "data" / "raw" / "landsat")
cfg["paths"]["sentinel_dir"]  = str(WORK_DIR / "data" / "raw" / "sentinel")
cfg["paths"]["aligned_dir"]   = str(WORK_DIR / "data" / "aligned")
cfg["paths"]["checkpoints"]   = str(WORK_DIR / "checkpoints")
cfg["paths"]["results"]       = str(WORK_DIR / "results")

# Kaggle CPU has many cores — use them
cfg["training"]["num_workers"] = 4
cfg["training"]["pin_memory"]  = False  # no GPU in Phase 1

with open(cfg_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print("✓ config.yaml patched for Kaggle paths:")
for k, v in cfg["paths"].items():
    print(f"  {k}: {v}")

---
## Step 5 — Download Satellite Data

Downloads 12 paired Landsat-8 + Sentinel-2 scenes for Rondônia, Brazil  
from **Microsoft Planetary Computer** (free, no account required).

**Storage:** ~4–6 GB total  
**Time:** ~20–40 minutes on Kaggle's network

In [ ]:
import subprocess, sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working")

result = subprocess.run(
    [sys.executable, str(WORK_DIR / "scripts" / "download_data.py"),
     "--config", str(WORK_DIR / "config.yaml")],
    cwd=str(WORK_DIR),
    capture_output=False,   # show output live
)

if result.returncode != 0:
    print("\n⚠️  download_data.py exited with errors. Check output above.")
else:
    # Count downloaded files
    ls = list((WORK_DIR / "data" / "raw" / "landsat").glob("*.tif"))
    ss = list((WORK_DIR / "data" / "raw" / "sentinel").glob("*.tif"))
    print(f"\n✓ Download complete.")
    print(f"  Landsat scenes  : {len(ls)}")
    print(f"  Sentinel scenes : {len(ss)}")

---
## Step 6 — Prepare Patches

Runs the full preprocessing pipeline:
1. **Histogram matching** (Landsat → Sentinel-2 spectral domain)
2. **Sub-pixel alignment** (phase cross-correlation)
3. **Patch extraction** (LR 48×48, HR 144×144, stride 24)
4. **RMSE filtering** (discards misaligned pairs > 0.3px)
5. **Pseudo-label generation** (NDVI-based forest/defor/road labels)

**Expected output:** ~2,000–3,000 `.npz` patch pairs across train/val/test

In [ ]:
import subprocess, sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working")

result = subprocess.run(
    [sys.executable, str(WORK_DIR / "scripts" / "prepare_patches.py"),
     "--config", str(WORK_DIR / "config.yaml")],
    cwd=str(WORK_DIR),
    capture_output=False,
)

if result.returncode != 0:
    print("\n⚠️  prepare_patches.py exited with errors.")
else:
    aligned = WORK_DIR / "data" / "aligned"
    for split in ["train", "val", "test"]:
        count = len(list((aligned / split).glob("*.npz")))
        print(f"  {split:5s}: {count} patches")
    print("\n✓ Patch preparation complete.")

---
## Step 7 — Quick Bicubic Baseline Verification

Runs the bicubic baseline evaluation (no GPU, no model weights needed).  
**Expected: PSNR ≥ 24 dB** — if you see this, the preprocessing fix is working.  
If PSNR is still ~16 dB, check the scene filename prefixes (T##, V##, TE##).

In [ ]:
import subprocess, sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working")

result = subprocess.run(
    [sys.executable, str(WORK_DIR / "evaluate.py"),
     "--model", "bicubic",
     "--scaling", "standard",
     "--test-split", "test",
     "--skip-lpips",   # skip LPIPS — no GPU and saves time
     "--config", str(WORK_DIR / "config.yaml")],
    cwd=str(WORK_DIR),
    capture_output=False,
)

if result.returncode != 0:
    print("⚠️  Evaluation failed. Check errors above.")

---
## Step 8 — Save Patches as Kaggle Dataset Output

Compresses the patches into a zip file in `/kaggle/working/`.  
After saving the notebook output, you can create a Kaggle Dataset from  
this notebook's output and attach it to Phase 2.

In [ ]:
import zipfile, os
from pathlib import Path
from tqdm import tqdm

WORK_DIR = Path("/kaggle/working")
aligned  = WORK_DIR / "data" / "aligned"
zip_path = WORK_DIR / "rondonia_sr_patches.zip"

all_npz = sorted(aligned.rglob("*.npz"))
print(f"Zipping {len(all_npz)} patch files...")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in tqdm(all_npz):
        arcname = f.relative_to(WORK_DIR)
        zf.write(f, arcname)

size_mb = zip_path.stat().st_size / 1e6
print(f"\n✓ Saved: {zip_path.name}  ({size_mb:.0f} MB)")
print("")
print("NEXT STEPS:")
print("  1. Click 'Save & Run All' (top right) to persist this notebook output")
print("  2. Go to: Output tab → find rondonia_sr_patches.zip")
print("  3. Click the 3-dot menu → 'Create Dataset from this output'")
print("  4. Name it: rondonia-sr-patches")
print("  5. Then open Phase 2 notebook and attach this dataset")

---
## ✅ Phase 1 Complete!

You now have ~2,700 aligned LR/HR patch pairs ready for training.

**What was done:**
- ✓ 12 Landsat-8 + Sentinel-2 scene pairs downloaded
- ✓ Histogram matching (sensor harmonization applied)
- ✓ Sub-pixel alignment (phase cross-correlation)
- ✓ Misaligned patches filtered (RMSE > 0.3px discarded)
- ✓ NDVI pseudo-labels generated for downstream segmentation
- ✓ Bicubic baseline verified (should be ≥ 24 dB)

**→ Open `phase2_train_evaluate.ipynb` next** (attach GPU, T4 or P100)